# Summary Evaluation Benchmark

|Metric | Scope Description |
|-----|------|
|BERTScore | Semantic similarity between LLaMA and GPT summaries |
|Hallucination Rate | Extracted numeric claims vs. metrics compute() output from FRED API |
|Key Fact Coverage | Whether annotated key data points appear in the answer (data-extraction questions only) |

Runs Summary Evaluations with 3 metrics across all 12 agent variants and saves results.

| # | Model | Retriever |
|---|-------|-----------|
| 1 | gpt-4o-mini | full guide |
| 2 | gpt-4o-mini | semantic |
| 3 | gpt-4o-mini (fine-tuned)| semantic + checks |
| 4 | llama3.2 | full guide |
| 5 | llama3.2 | semantic |
| 6 | llama3.2 | semantic + date parser + checks |
| 7 | llama3.2 (fine-tuned)| full guide |
| 8 | llama3.2 (fine-tuned)| semantic |
| 9 | llama3.2 (fine-tuned)| semantic + date parser + checks |
| 10 | llama3.2 (few-shot)| full guide |
| 11 | llama3.2 (few-shot)| semantic |
| 12 | llama3.2 (few-shot)| semantic + date parser + checks |

In [2]:
from summary_evaluation import evaluate_summary_factuality
import json
from bert_score import score
import os
import pandas as pd

GPT_DIR = "files/gpt-4o-mini"
LLAMA_DIR = "files/llama3.2"
TEST_FILE = "data/QA_test.json"
GPT_REFERENCE = "files/gpt-4o-mini/QA_test_gpt_api_final.json"

In [3]:
os.makedirs(GPT_DIR + "/summary_eval", exist_ok=True)
os.makedirs(LLAMA_DIR + "/summary_eval", exist_ok=True)

In [4]:
def run_all_tests(
    api_file_path,
    out_path,
    test_file_path=TEST_FILE,
    gpt_ref_path=GPT_REFERENCE,
    compute_bertscore=True,
):
    """
    Run all summary evaluations on a single agent result file.

    Args:
        api_file_path:     Path to the agent result JSON (list of result dicts).
        out_path:          Path to save the evaluation result JSON.
        test_file_path:    Path to QA_test.json.
        gpt_ref_path:      Path to GPT reference answers (used for BERTScore).
        compute_bertscore: Whether to compute BERTScore (LLaMA only, default True).

    Returns:
        dict with avg_key_fact_coverage, avg_hallucination_rate,
        and avg_bertscore (if compute_bertscore=True).
    """
    with open(test_file_path, encoding="utf-8") as f:
        test_cases = json.load(f)
    with open(api_file_path, encoding="utf-8") as f:
        api_results_all = json.load(f)

    # load GPT reference only when needed
    gpt_answers = []
    if compute_bertscore:
        with open(gpt_ref_path, encoding="utf-8") as f:
            gpt_file = json.load(f)
        gpt_answers = [r.get("final_answer", "") for r in gpt_file]

    # print(f"\n{'#'*60}")
    # print(f"Running {len(test_cases)} test cases...")
    # print(f"{'#'*60}")

    per_question_results = []
    cand_answers = []   # for BERTScore, aligned with gpt_answers
    ref_answers  = []

    for i, test_case in enumerate(test_cases):
        # print(f"\n[Test {i+1}/{len(test_cases)}] {test_case.get('question_id', '')}")

        result_entry = api_results_all[i] if i < len(api_results_all) else {}

        answer     = result_entry.get("final_answer", "")
        api_results = result_entry.get("api_results", [])
        key_facts  = test_case.get("key_facts", None)

        factuality = evaluate_summary_factuality(
            answer=answer,
            api_results=api_results,
            key_facts=key_facts,
            compact=True
        )
        kf = factuality["key_fact_coverage"]
        nf = factuality["numeric_faithfulness"]

        per_question_results.append({
            "question_id":        test_case.get("question_id", i),
            "question":           test_case.get("question", ""),
            "key_fact_coverage":  kf,
            "hallucination_rate": nf,
        })

        # collect for BERTScore — skip failed/empty answers
        if compute_bertscore and answer:
            cand_answers.append(answer)
            ref_answers.append(gpt_answers[i] if i < len(gpt_answers) else "")

    kf_scores = [r["key_fact_coverage"]  for r in per_question_results]
    nf_scores = [r["hallucination_rate"] for r in per_question_results]
    avg_kf = sum(kf_scores) / len(kf_scores) if kf_scores else 0.0
    avg_nf = sum(nf_scores) / len(nf_scores) if nf_scores else 0.0

    print(f"\nKey Fact Coverage Rate : {avg_kf:.4f}")
    print(f"Hallucination Rate     : {avg_nf:.4f}")

    summary = {
        "avg_key_fact_coverage":  round(avg_kf, 4),
        "avg_hallucination_rate": round(avg_nf, 4),
    }

    if compute_bertscore and cand_answers:
        # only score pairs where both sides are non-empty
        MAX_CHARS = 512 * 4  # ~512 tokens at ~4 chars/token
        valid_pairs = [
            (c[:MAX_CHARS], r[:MAX_CHARS])
            for c, r in zip(cand_answers, ref_answers)
            if c and r
        ]
        valid_ids = [
            tc.get("question_id", i)
            for i, (tc, c, r) in enumerate(zip(test_cases, cand_answers, ref_answers))
            if c and r
        ]
        if valid_pairs:
            cands, refs = zip(*valid_pairs)
            _, _, F1 = score(
                cands=list(cands),
                refs=list(refs),
                lang="en",
                model_type="roberta-large",  # 512 token limit, stable
                batch_size=8,
                verbose=False
            )
            avg_bs = round(F1.mean().item(), 4)
            bs_list = F1.tolist()
        else:
            avg_bs = 0.0
            bs_list = []

        print(f"BERTScore (F1)         : {avg_bs:.4f}")
        summary["avg_bertscore"] = avg_bs

        # attach per-question BERTScore
        id_to_bs = dict(zip(valid_ids, bs_list))
        for i, r in enumerate(per_question_results):
            qid = r.get("question_id", i)
            r["bertscore"] = round(id_to_bs[qid], 4) if qid in id_to_bs else None

    # save results
    output = {"summary": summary, "results": per_question_results}
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(output, f, indent=2, ensure_ascii=False)
    print(f"\nResults saved to {out_path}")

    return summary

---
## 1. GPT-4o-mini — Baseline (full guide)

In [5]:
res_gpt_base = run_all_tests(
    api_file_path=f"{GPT_DIR}/QA_test_gpt_api.json",
    out_path=f"{GPT_DIR}/summary_eval/gpt_api.json",
    compute_bertscore=False
)


Key Fact Coverage Rate : 0.9600
Hallucination Rate     : 0.3700

Results saved to files/gpt-4o-mini/summary_eval/gpt_api.json


---
## 2. GPT-4o-mini — Semantic Retriever

In [6]:
res_gpt_sem = run_all_tests(
    api_file_path=f"{GPT_DIR}/QA_test_gpt_api_semantic_retriever.json",
    out_path=f"{GPT_DIR}/summary_eval/gpt_api_semantic_retriever.json",
    compute_bertscore=False
)


Key Fact Coverage Rate : 0.9717
Hallucination Rate     : 0.3643

Results saved to files/gpt-4o-mini/summary_eval/gpt_api_semantic_retriever.json


---
## 3. GPT-4o-mini — Final (semantic + date parser + checks)

In [7]:
res_gpt_final = run_all_tests(
    api_file_path=f"{GPT_DIR}/QA_test_gpt_api_final.json",
    out_path=f"{GPT_DIR}/summary_eval/gpt_api_final.json",
    compute_bertscore=False
)


Key Fact Coverage Rate : 0.9717
Hallucination Rate     : 0.3542

Results saved to files/gpt-4o-mini/summary_eval/gpt_api_final.json


---
## 4. LLaMA 3.2 — Baseline (full guide)

In [8]:
res_llama_base = run_all_tests(
    api_file_path=f"{LLAMA_DIR}/QA_test_llama_api.json",
    out_path=f"{LLAMA_DIR}/summary_eval/llama_api.json",
    compute_bertscore=True
)


Key Fact Coverage Rate : 0.9283
Hallucination Rate     : 0.5883


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore (F1)         : 0.8427

Results saved to files/llama3.2/summary_eval/llama_api.json


---
## 5. LLaMA 3.2 — Semantic Retriever

In [9]:
res_llama_sem = run_all_tests(
    api_file_path=f"{LLAMA_DIR}/QA_test_llama_api_semantic_retriever.json",
    out_path=f"{LLAMA_DIR}/summary_eval/llama_api_semantic_retriever.json",
    compute_bertscore=True
)


Key Fact Coverage Rate : 0.9283
Hallucination Rate     : 0.4702


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore (F1)         : 0.8467

Results saved to files/llama3.2/summary_eval/llama_api_semantic_retriever.json


---
## 6. LLaMA 3.2 — Final (semantic + date parser + checks)

In [10]:
res_llama_final = run_all_tests(
    api_file_path=f"{LLAMA_DIR}/QA_test_llama_api_final.json",
    out_path=f"{LLAMA_DIR}/summary_eval/llama_api_final.json",
    compute_bertscore=True
)


Key Fact Coverage Rate : 0.9333
Hallucination Rate     : 0.4945


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore (F1)         : 0.8478

Results saved to files/llama3.2/summary_eval/llama_api_final.json


---
## 7. LLaMA 3.2 — Fine-tuned variants

Update the file paths below to point to your fine-tuned model result files.

In [11]:
# tool + summary
res_llama_ft_base3 = run_all_tests(
    api_file_path=f"{LLAMA_DIR}/QA_test_llama_api_finetuned3.json",
    out_path=f"{LLAMA_DIR}/summary_eval/llama_api_finetuned3.json",
    compute_bertscore=True
)

res_llama_ft_sem3 = run_all_tests(
    api_file_path=f"{LLAMA_DIR}/QA_test_llama_api_semantic_retriever_finetuned3.json",
    out_path=f"{LLAMA_DIR}/summary_eval/llama_api_semantic_retriever_finetuned3.json",
    compute_bertscore=True
)

res_llama_ft_final3 = run_all_tests(
    api_file_path=f"{LLAMA_DIR}/QA_test_llama_api_final_finetuned3.json",
    out_path=f"{LLAMA_DIR}/summary_eval/llama_api_final_finetuned3.json",
    compute_bertscore=True
)


Key Fact Coverage Rate : 0.8833
Hallucination Rate     : 0.5961


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore (F1)         : 0.8241

Results saved to files/llama3.2/summary_eval/llama_api_finetuned3.json

Key Fact Coverage Rate : 0.8867
Hallucination Rate     : 0.5987


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore (F1)         : 0.8284

Results saved to files/llama3.2/summary_eval/llama_api_semantic_retriever_finetuned3.json

Key Fact Coverage Rate : 0.9150
Hallucination Rate     : 0.5897


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore (F1)         : 0.8369

Results saved to files/llama3.2/summary_eval/llama_api_final_finetuned3.json


In [12]:
# # summary only
# res_llama_ft_base4 = run_all_tests(
#     api_file_path=f"{LLAMA_DIR}/QA_test_llama_api_finetuned4.json",
#     out_path=f"{LLAMA_DIR}/summary_eval/llama_api_finetuned4.json",
#     compute_bertscore=True
# )

# res_llama_ft_sem4 = run_all_tests(
#     api_file_path=f"{LLAMA_DIR}/QA_test_llama_api_semantic_retriever_finetuned4.json",
#     out_path=f"{LLAMA_DIR}/summary_eval/llama_api_semantic_retriever_finetuned4.json",
#     compute_bertscore=True
# )

# res_llama_ft_final4 = run_all_tests(
#     api_file_path=f"{LLAMA_DIR}/QA_test_llama_api_final_finetuned4.json",
#     out_path=f"{LLAMA_DIR}/summary_eval/llama_api_final_finetuned4.json",
#     compute_bertscore=True
# )

## 8. Few-shot Prompt

In [13]:
res_llama_fs_base = run_all_tests(
    api_file_path=f"{LLAMA_DIR}/QA_test_llama_api_few_shot.json",
    out_path=f"{LLAMA_DIR}/summary_eval/llama_api_few_shot.json",
    compute_bertscore=True
)

res_llama_fs_sem = run_all_tests(
    api_file_path=f"{LLAMA_DIR}/QA_test_llama_api_semantic_retriever_few_shot.json",
    out_path=f"{LLAMA_DIR}/summary_eval/llama_api_semantic_retriever_few_shot.json",
    compute_bertscore=True
)

res_llama_fs_final = run_all_tests(
    api_file_path=f"{LLAMA_DIR}/QA_test_llama_api_final_few_shot.json",
    out_path=f"{LLAMA_DIR}/summary_eval/llama_api_final_few_shot.json",
    compute_bertscore=True
)


Key Fact Coverage Rate : 0.8900
Hallucination Rate     : 0.5590


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore (F1)         : 0.8415

Results saved to files/llama3.2/summary_eval/llama_api_few_shot.json

Key Fact Coverage Rate : 0.9133
Hallucination Rate     : 0.4976


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore (F1)         : 0.8456

Results saved to files/llama3.2/summary_eval/llama_api_semantic_retriever_few_shot.json

Key Fact Coverage Rate : 0.9383
Hallucination Rate     : 0.4491


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERTScore (F1)         : 0.8471

Results saved to files/llama3.2/summary_eval/llama_api_final_few_shot.json


---
## 9. Summary Comparison

In [14]:
rows = [
    ("gpt_api",                        res_gpt_base),
    ("gpt_api_semantic_retriever",      res_gpt_sem),
    ("gpt_api_final",                   res_gpt_final),
    ("llama_api",                       res_llama_base),
    ("llama_api_semantic_retriever",    res_llama_sem),
    ("llama_api_final",                 res_llama_final),
    ("llama_api_finetuned3",             res_llama_ft_base3),
    ("llama_api_semantic_finetuned3",    res_llama_ft_sem3),
    ("llama_api_final_finetuned3",       res_llama_ft_final3),
    # ("llama_api_finetuned4",             res_llama_ft_base4),
    # ("llama_api_semantic_finetuned4",    res_llama_ft_sem4),
    # ("llama_api_final_finetuned4",       res_llama_ft_final4),
    ("llama_api_few_shot",             res_llama_fs_base),
    ("llama_api_semantic_few_shot",    res_llama_fs_sem),
    ("llama_api_final_few_shot",       res_llama_fs_final),
]

records = []
for name, res in rows:
    records.append({
        "agent":                name,
        "key_fact_coverage":   res.get("avg_key_fact_coverage",  "-"),
        "hallucination_rate":  res.get("avg_hallucination_rate", "-"),
        "bertscore_f1":        res.get("avg_bertscore",          "N/A"),
    })

df = pd.DataFrame(records).set_index("agent")
df

,key_fact_coverage,hallucination_rate,bertscore_f1
agent,,,
gpt_api,0.9600,0.3700,N/A
gpt_api_semantic_retriever,0.9717,0.3643,N/A
gpt_api_final,0.9717,0.3542,N/A
llama_api,0.9283,0.5883,0.8427
llama_api_semantic_retriever,0.9283,0.4702,0.8467
llama_api_final,0.9333,0.4945,0.8478
llama_api_finetuned3,0.8833,0.5961,0.8241
llama_api_semantic_finetuned3,0.8867,0.5987,0.8284
llama_api_final_finetuned3,0.9150,0.5897,0.8369
